# 04 · Deep Agent (v2) — the SOC deep-agent harness

**Vertical:** Cybersecurity · **Use case:** alert-triage · **ADLC stage:** Deep Agent (capstone)

Labs 01-03 built a single ReAct agent. A **deep agent** isn't just "an agent with sub-agents" — it's a
**managed harness**: a filesystem workspace, a planner, pluggable backends + persistent memory,
middleware, human approval gates, a file-based identity, and automatic context management — all built
in. This capstone adds those pieces **one at a time**, SOC-flavored, then assembles them into a
campaign-triage coordinator and **measures** it against the single agent.

![Deep agent](images/04_deep_agent.png)

| Piece | API | SOC use |
|---|---|---|
| Harness (filesystem + planner) | `create_deep_agent(model=…)` | a triage scratchpad + `write_todos` plan |
| Domain tools | `tools=[…]` | the Lab 01 alert / asset / history / KB tools |
| Sub-agent (isolated delegation) | `subagents=[…]` + `task` | investigate one alert in a clean context |
| Backends & memory | `StateBackend` / `FilesystemBackend` / `StoreBackend` / `CompositeBackend` | persist analyst notes across cases |
| Middleware | `@wrap_tool_call` | a tool-call **audit log** |
| HITL | `interrupt_on={…}` | **approve before** the agent commits a decision |
| Identity + skills | `memory=["/AGENTS.md"]`, `skills=["/skills/"]` | coordinator persona + correlation playbook |
| Typed output + eval | `response_format=` + LangSmith `evaluate` | typed verdict + a measured deep-agent eval |

> Self-directed module · ~35 min. By the end you can:
> - build a deep agent and use its **free harness** — filesystem tools, `write_todos`, context management
> - add **backends & cross-thread memory**, **middleware**, and **HITL** one kwarg at a time
> - give it a **file-based identity + skill**, then **measure** the deep agent on a campaign dataset
>
> 🧭 **Builds on Labs 01-03; runs standalone.** Data is synthetic. The lesson: a deep agent is a
> **harness** — that, not a scoreboard win, is why you reach for one.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # FIRST — before any model client is constructed
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| tracing:", os.getenv("LANGSMITH_TRACING"))

## 1. Your first deep agent — the harness

`create_deep_agent(model=…)` with **nothing else** already ships a working harness. For free, the agent
gets:

| Built-in | What it is |
|---|---|
| Filesystem tools | `ls` · `read_file` · `write_file` · `edit_file` · `glob` · `grep` — a scratchpad |
| Planner | `write_todos` — the agent plans multi-step work |
| Delegation | `task` — hand work to sub-agents (added in §3) |
| Context management | auto-summarize near capacity + offload large tool results to files (see §5) |

In [ ]:
from deepagents import create_deep_agent

from utils.models import get_model

harness = create_deep_agent(model=get_model())  # no tools, no sub-agents — just the harness

The agent can already keep a **triage scratchpad** — write intermediate notes to a file and read them
back — without any tool we wrote.

In [ ]:
out = harness.invoke({"messages": [{"role": "user", "content":
    "Write 'working notes: checking ALT-2003' to /scratch.md, then read it back and confirm the contents."}]})
print(out["messages"][-1].content[:300])

> 💡 A single ReAct agent (Labs 01-03) has **none** of this — no filesystem, no planner, no context
> management. That harness is the real reason to reach for a deep agent, before we add a single sub-agent.

## 2. Give it the triage tools

The harness can plan and take notes, but it can't see SOC data yet. Add Lab 01's grounding tools —
imported as plumbing (taught in `01_build`), passed with one `tools=` kwarg.

In [ ]:
from alert_triage_graph import (
    AGENT_DIR,
    CampaignTriageReport,
    PROMPT_NAME,
    PROMPT_PATH,
    get_alert,
    get_asset_context,
    search_incident_history,
    search_soc_kb,
)
from utils.prompts import dataset_url, get_prompt, prompt_url

TRIAGE_TOOLS = [get_alert, get_asset_context, search_incident_history, search_soc_kb]
base_prompt = get_prompt(PROMPT_NAME, fallback=PROMPT_PATH.read_text().strip())

triage_agent = create_deep_agent(
    model=get_model(),
    system_prompt=base_prompt,
    tools=TRIAGE_TOOLS,  # ← only addition vs §1: the SOC grounding tools
)
from utils.trace import ask  # .invoke + a printed trace link

print(ask(triage_agent, "Triage ALT-2003 and give a one-line disposition.")[:300])

## 3. Sub-agents — isolated delegation

A multi-alert **campaign** is many investigations plus a correlation. Rather than triage every alert in
one context, delegate each to a scoped **sub-agent** and let a **coordinator** correlate. A sub-agent is
a plain dict; the `description` is what the coordinator reads to decide when to delegate.

![Coordinator + sub-agent](images/04_coordinator.png)

In [ ]:
investigator = {
    "name": "alert-investigator",
    "description": (
        "Fully investigate ONE alert: retrieve it, enrich every asset, and match prior incidents + SOC "
        "guidance. Report cited evidence and the accounts/IPs/hosts/domains it touches. Do not decide the campaign."
    ),
    "system_prompt": (
        "You investigate exactly one alert end to end. Use get_alert, then get_asset_context for every "
        "asset, then search_incident_history and search_soc_kb. Report cited evidence and the entities it "
        "involves. Do not decide whether alerts form a campaign — that is the coordinator's job."
    ),
    "tools": TRIAGE_TOOLS,
}

> 💡 The coordinator gets **no leaf tools** — only the built-in `task`. That forces it to *delegate* each
> alert to `alert-investigator` (isolated context) and then correlate, instead of triaging inline. Each
> delegation is a labeled sub-investigation you can inspect in the trace.

In [ ]:
coordinator_prompt = (
    base_prompt
    + "\n\nYou coordinate multi-alert campaign investigations. Delegate each alert to the "
    "alert-investigator sub-agent, correlate the findings on shared entities, exclude decoys, and "
    "synthesize one verdict. Do not invent facts absent from sub-agent results."
)

coordinator = create_deep_agent(
    model=get_model(),
    system_prompt=coordinator_prompt,
    subagents=[investigator],  # ← the coordinator delegates to this (no leaf tools of its own)
)


def show_graph(agent) -> None:  # deep-agent graphs 400 via mermaid.ink — list nodes/edges instead
    g = agent.get_graph()
    print("nodes:", ", ".join(sorted(n for n in g.nodes)))


show_graph(coordinator)

## 4. Backends & memory — what persists, and where

A **backend** decides where the agent's files live; **memory** decides what survives across threads.

| Backend | Scope |
|---|---|
| `StateBackend` (default) | ephemeral — files vanish when the run/thread ends |
| `FilesystemBackend(root_dir=…)` | real disk — how we serve `AGENTS.md` + skills (§7) |
| `StoreBackend` (+ a `store`) | durable key-value, survives across threads |
| `CompositeBackend(default=…, routes={…})` | route some paths (e.g. `/memories/`) to a durable store |

Route `/memories/` to a `StoreBackend` and the agent remembers **across cases (threads)** — write a
prior-campaign note in one thread, recall it in another. Pass an explicit `namespace` to scope that
memory (here to a tenant; you'd key it per-user/tenant in production).

![Backends & memory](images/04_backends.png)

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

memory_agent = create_deep_agent(
    model=get_model(),
    system_prompt=base_prompt,
    tools=TRIAGE_TOOLS,
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=lambda rt: ("meridian-union", "soc-memories"))},  # ← route + scope
    ),
    store=InMemoryStore(),          # ← durable store behind the /memories/ route
    checkpointer=InMemorySaver(),   # ← thread continuity
)

case_a = {"configurable": {"thread_id": "case-A"}}
case_b = {"configurable": {"thread_id": "case-B"}}
memory_agent.invoke({"messages": [{"role": "user", "content":
    "Save this to /memories/prior.md exactly: 'acct-fin-controller-x was involved in a prior finance campaign'."}]}, config=case_a)
recall = memory_agent.invoke({"messages": [{"role": "user", "content":
    "Read /memories/prior.md. What account does it flag?"}]}, config=case_b)  # different thread!
print(recall["messages"][-1].content[:200])

## 5. Middleware — a SOC audit log

Middleware wraps the agent's model/tool calls with pluggable behavior. Deep agents already stack several
built-ins (todos, filesystem, sub-agents, **summarization**, patch-tool-calls). You add your own — for a
SOC, an **audit log** of every tool call is the obvious one.

In [ ]:
from langchain.agents.middleware import wrap_tool_call

AUDIT: list[str] = []


@wrap_tool_call
def audit_log(request, handler):
    call = getattr(request, "tool_call", None) or {}
    AUDIT.append(call.get("name", "?") if isinstance(call, dict) else str(call))
    return handler(request)  # then run the tool as normal


audited = create_deep_agent(model=get_model(), system_prompt=base_prompt, tools=TRIAGE_TOOLS, middleware=[audit_log])
audited.invoke({"messages": [{"role": "user", "content": "Triage ALT-2003."}]})
print("audited tool calls:", AUDIT)

> 💡 **Context management is built-in middleware.** When history nears ~85% of the context window the
> agent auto-summarizes, and tool results larger than ~20k tokens are offloaded to the filesystem instead
> of bloating the prompt. That's why a deep agent scales to many alerts/sources where a single flat
> context degrades — no code required.

![Context management](images/04_context_mgmt.png)

## 6. Human-in-the-loop — approve before committing a decision

`interrupt_on` pauses the agent before a chosen tool so a human can approve. In a SOC you gate the
moment the agent **commits a decision to the case file** (a `write_file`): it pauses, you approve, it
resumes.

`invoke → ⏸️ interrupt → decide → Command(resume=…) → ▶️ continue`

In [ ]:
from langgraph.types import Command

approver = create_deep_agent(
    model=get_model(),
    system_prompt=base_prompt,
    tools=TRIAGE_TOOLS,
    interrupt_on={"write_file": True},  # ← pause before any write to the case file
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "hitl-demo"}}
paused = approver.invoke({"messages": [{"role": "user", "content":
    "Triage ALT-2003, then write your one-line decision to /case-notes/ALT-2003.md."}]}, config=cfg)
print("⏸️ interrupted:", "__interrupt__" in paused)

In [ ]:
resumed = approver.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg)  # ▶️ human approves
print("▶️ completed:", "__interrupt__" not in resumed)

## 7. File-based identity + a skill

Instead of one giant system prompt, a deep agent loads an editable **`AGENTS.md`** (identity, read at
startup) and **skills** (`SKILL.md`, loaded on demand). We serve both from a `FilesystemBackend` rooted
at the use case's `data/agent/` folder.

In [ ]:
print((AGENT_DIR / "AGENTS.md").read_text())
print("--- skill ---")
print((AGENT_DIR / "skills" / "campaign-correlation" / "SKILL.md").read_text()[:400], "…")

## 8. The complete campaign agent

Assemble it: coordinator + one investigator sub-agent, `AGENTS.md` identity + skill from a
`FilesystemBackend`, and a typed **`CampaignTriageReport`** (linked alerts, pivots, aggregate severity —
the schema is code, imported from Lab 01's graph module). Memory, middleware, and HITL from §4-6 compose
in the same way when a deployment needs them.

![Complete agent](images/04_complete.png)

In [ ]:
from deepagents.backends import FilesystemBackend

deep_agent = create_deep_agent(
    model=get_model(),
    system_prompt=coordinator_prompt,
    subagents=[investigator],
    backend=FilesystemBackend(root_dir=str(AGENT_DIR), virtual_mode=True),  # serves AGENTS.md + skills/
    memory=["/AGENTS.md"],   # file-based coordinator identity
    skills=["/skills/"],     # on-demand campaign-correlation skill
    response_format=CampaignTriageReport,  # typed campaign verdict
)

Run the cluster — four linked alerts (an account-takeover kill-chain on one source IP) plus a look-alike
decoy. Watch the trace: one `task` per alert, then the correlation.

In [ ]:
CAMPAIGN_ALERTS = ["ALT-2004", "ALT-2005", "ALT-2006", "ALT-2007", "ALT-2009"]  # 4 linked + 1 decoy
report = ask(deep_agent,
             "Triage these five alerts together as one investigation: " + ", ".join(CAMPAIGN_ALERTS) + ". "
             "Decide whether they are a coordinated campaign, exclude any decoy, and give the shared "
             "pivots, an aggregate severity/escalation, and one containment plan.")

In [ ]:
print(f"coordinated campaign : {report.is_coordinated_campaign}")
print(f"aggregate severity   : {report.aggregate_severity}  | escalate: {report.aggregate_escalate}")
print(f"linked alerts        : {', '.join(report.linked_alert_ids)}")
print(f"excluded (decoys)    : {', '.join(report.excluded_alert_ids)}")
print(f"pivot entities       : {', '.join(report.pivot_entities)}")
print("\ncitations:", ", ".join(report.citations))

## 9. Open it in LangGraph Studio

The same call is packaged in `alert_triage_graph.py` as `graph_deep_v2` and registered in
`langgraph.json`. From `content/`:

```bash
uv run langgraph dev     # serves at http://127.0.0.1:2024
```
Studio: **https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024**

| Graph | Try in Studio |
|---|---|
| `alert_triage_campaign_agent` | *"Triage ALT-2004, ALT-2005, ALT-2006, ALT-2007, ALT-2009 as one investigation; exclude decoys."* → the trace shows one `task` per alert |

## 10. Measure the deep agent

01-style tutorials *assert* the deep agent's value; we **measure** it — run the deep agent on a campaign
dataset. Two evaluators, each returning `{"key", "score", "comment"}`:

| Evaluator | Kind | Checks |
|---|---|---|
| `campaign_correctness` | reference-based judge | linked set, decoy exclusion, aggregate severity/escalation |
| `subagent_delegations` | trace-based | `task` hand-offs — the deep-agent signal earlier labs don't have |

In [ ]:
from evals.seed_alert_triage_campaign import DATASET_NAME, DATASET_PATH, seed_dataset

seed_dataset()  # idempotent push; pass refresh=True after editing the JSON to update the remote copy
print("📊 dataset:", dataset_url(DATASET_NAME))

In [ ]:
import json

example = json.loads(DATASET_PATH.read_text())[0]
print("Q         :", example["inputs"]["messages"][-1]["content"][:150], "…")
print("reference :", example["outputs"]["reference_answer"][:280], "…")

### The eval target

The target reuses the **§8 deep agent** — we grade what Studio and a deployment run. Copy the input
messages: a deep agent normalizes them in place, which would otherwise mutate the shared eval example.

In [ ]:
def run_deep(inputs: dict) -> dict:
    messages = [dict(m) for m in inputs["messages"]]  # avoid mutating the shared eval example
    return {"answer": deep_agent.invoke({"messages": messages})["structured_response"].model_dump_json(indent=2)}

### Evaluator #1: campaign_correctness

Reference-based. The full judge prompt is a visible constant — it grades the **correlation**, not
per-alert enums (which both agents get right).

In [ ]:
from pydantic import BaseModel, Field

CAMPAIGN_CORRECTNESS_PROMPT = """You grade a SOC campaign-triage answer against a reference correlation.

Return True only if the answer gets the CAMPAIGN decision right: the coordinated-vs-independent call,
the set of linked alert IDs, the excluded decoys, and the aggregate severity/escalation. Allow correct
extra detail. Fail for a wrong linked set, a decoy left in, missing the coordinated call, or a wrong
aggregate severity/escalation.

<question>{question}</question>
<answer>{answer}</answer>
<reference_answer>{reference}</reference_answer>
<reference_is_coordinated>{coordinated}</reference_is_coordinated>
<reference_linked>{linked}</reference_linked>
<reference_excluded>{excluded}</reference_excluded>
<reference_aggregate_severity>{severity}</reference_aggregate_severity>"""


class CampaignScore(BaseModel):
    reasoning: str = Field(description="Which correlation elements matched or missed.")
    score: bool = Field(description="True if the campaign decision is correct.")


def campaign_correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judge = get_model().with_structured_output(CampaignScore)
    v = judge.invoke(CAMPAIGN_CORRECTNESS_PROMPT.format(
        question=inputs["messages"][-1]["content"],
        answer=outputs["answer"],
        reference=reference_outputs["reference_answer"],
        coordinated=reference_outputs["is_coordinated_campaign"],
        linked=", ".join(reference_outputs["linked_alert_ids"]) or "none",
        excluded=", ".join(reference_outputs["excluded_alert_ids"]) or "none",
        severity=reference_outputs["aggregate_severity"],
    ))
    return {"key": "campaign_correctness", "score": v.score, "comment": v.reasoning}

### Evaluator #2: subagent_delegations

Trace-based — walks the `run` tree and counts the coordinator's `task` calls. A single agent scores
**0** here structurally; the deep agent's count is the visible proof it decomposed the work.

In [ ]:
from langsmith.schemas import Run


def subagent_delegations_evaluator(run: Run) -> dict:
    def _count(node) -> int:
        name = (getattr(node, "name", "") or "").lower()
        n = 1 if getattr(node, "run_type", None) == "tool" and "task" in name else 0
        for child in getattr(node, "child_runs", None) or []:
            n += _count(child)
        return n

    return {"key": "subagent_delegations", "score": _count(run), "comment": "task hand-offs in the run tree."}

### Dry-run one example first

In [ ]:
outputs = run_deep(example["inputs"])
print("campaign_correctness:", campaign_correctness_evaluator(example["inputs"], outputs, example["outputs"]))

### Scale to the dataset → scorecard

In [ ]:
from collections import defaultdict

from langsmith import evaluate

experiment = evaluate(
    run_deep,
    data=DATASET_NAME,
    evaluators=[campaign_correctness_evaluator, subagent_delegations_evaluator],
    experiment_prefix="alert-triage-04v2-deep",
    max_concurrency=2,
)

scores: dict[str, list] = defaultdict(list)
for item in experiment:
    for r in item["evaluation_results"]["results"]:
        scores[r.key].append(r.score)

print("\n📊 Scorecard — deep agent")
for key, vals in scores.items():
    if vals and all(isinstance(v, bool) for v in vals):
        print(f"  {key:22s}: {sum(vals)}/{len(vals)} passed")
    else:
        nums = [v for v in vals if isinstance(v, (int, float))]
        print(f"  {key:22s}: avg {sum(nums) / len(nums):.2f}" if nums else f"  {key}: n/a")
print("\n📝 prompt:", prompt_url(PROMPT_NAME))

**Read the scorecard honestly.** The deep agent triages the campaign correctly, and
**`subagent_delegations`** shows *how* — each alert was a traceable, decomposed sub-investigation (a
signal earlier labs don't have). The value here isn't a higher score than a plain agent — a capable
model correlates a handful of alerts in one context about as well — it's everything the harness gives
you around the answer: the filesystem workspace, cross-thread memory, audit-log middleware, approval gates, and
automatic context management that let it **scale**.

> 💡 The capstone lesson: a deep agent is a **harness**, not a scoreboard trick. Reach for it when you
> need legibility, memory, approval gates, and scale — then **measure** to confirm it earns its keep.

## 11. Recap & next

| Topic | API |
|---|---|
| Deep-agent harness | `create_deep_agent(model=…)` → filesystem tools + `write_todos` + context mgmt |
| Domain tools / sub-agent | `tools=[…]` · `subagents=[{…}]` + `task` |
| Backends & memory | `StateBackend` / `FilesystemBackend` / `StoreBackend` / `CompositeBackend(routes={"/memories/":…})` |
| Middleware | `@wrap_tool_call` (audit log) |
| HITL | `interrupt_on={"write_file": True}` + `Command(resume=…)` |
| Identity + skills | `memory=["/AGENTS.md"]`, `skills=["/skills/"]` on a `FilesystemBackend` |
| Typed output + eval | `response_format=CampaignTriageReport` + trace-based delegation evaluator |

The arc: **built** (01) → **evaluated** (02) → **orchestrated** (03) → **deep-agent harness** (04).
Swap the synthetic fixtures for a customer's SOC corpus and the shape holds.

**Next:** `05_observe.ipynb` — take it to production: score live triage traces with online run rules and
route low scores to an annotation queue.